# Import libraries

In [ ]:
import numpy as np
import pandas as pd

import concurrent.futures

pd.set_option('max_rows', 300)
pd.set_option('max_columns', 300)

from tqdm.notebook import tqdm

In [ ]:
! ls -hlt ../input/optiver-realized-volatility-prediction/book_train.parquet | wc -l

# Custom functions

In [ ]:
def log_return(list_stock_prices):
    return np.log(list_stock_prices).diff()

In [ ]:
def realized_volatility(series_log_return):
    return np.sqrt(np.sum(series_log_return**2))

# Explore data

In [ ]:
train_df = pd.read_csv("/kaggle/input/optiver-realized-volatility-prediction/train.csv")
test_df = pd.read_csv("/kaggle/input/optiver-realized-volatility-prediction/test.csv")
submit_df = pd.read_csv("/kaggle/input/optiver-realized-volatility-prediction/sample_submission.csv")

print(train_df.shape)
print(test_df.shape)
print(submit_df.shape)

In [ ]:
train_df.head(2)

In [ ]:
print(f"Unique stock ids: {train_df['stock_id'].nunique()}")
print(f"Unique time ids: {train_df['time_id'].nunique()}")

In [ ]:
stockLst = train_df['stock_id'].unique().tolist()

In [ ]:
book_df = pd.read_parquet(f"../input/optiver-realized-volatility-prediction/book_train.parquet/stock_id={stockLst[0]}")

print(book_df.shape)

book_df.head()

In [ ]:
book_df['wap'] = (book_df['bid_price1'] * book_df['ask_size1'] +
                                book_df['ask_price1'] * book_df['bid_size1']) / (
                                       book_df['bid_size1']+ book_df['ask_size1'])


# Log return
book_df.loc[:,'log_return'] = log_return(book_df['wap'])
book_df = book_df[~book_df['log_return'].isnull()]

realized_vol = realized_volatility(book_df[book_df['time_id']==5]['log_return'])

print(f'Realized volatility for stock_id 0 on time_id 5 is {realized_vol}')

# Feature Engineering

In [ ]:
def order_book_features(stock):
    # Read in order book for stock
    book_df = pd.read_parquet(f"/kaggle/input/optiver-realized-volatility-prediction/book_train.parquet/stock_id={stock}")
    
    # WAP
    book_df['wap'] = (book_df['bid_price1'] * book_df['ask_size1'] +
                                book_df['ask_price1'] * book_df['bid_size1']) / (
                                       book_df['bid_size1']+ book_df['ask_size1'])
    book_df['wap2'] = (book_df['bid_price2'] * book_df['ask_size2'] +
                                book_df['ask_price2'] * book_df['bid_size2']) / (
                                       book_df['bid_size2']+ book_df['ask_size2'])
    
    # Spread
    book_df['spread'] = (book_df['ask_price1'] / book_df['bid_price1']) - 1
    book_df['spread2'] = (book_df['ask_price2'] / book_df['bid_price2']) - 1
    
    # Bid-ask diff
    book_df['bid_ask_diff'] = (book_df['ask_price1'] - book_df['bid_price1'])
    book_df['bid_ask_diff2'] = (book_df['ask_price2'] - book_df['bid_price2'])
    book_df['bid_ask_diff_wap_ratio'] = (book_df['ask_price1'] - book_df['bid_price1']) / book_df['wap']
    book_df['bid_ask_diff_wap2_ratio'] = (book_df['ask_price1'] - book_df['bid_price1']) / book_df['wap2']
    book_df['bid_ask_diff2_wap2_ratio'] = (book_df['ask_price2'] - book_df['bid_price2']) / book_df['wap2']
    
    # Log return
    book_df['log_return'] = book_df.groupby(['time_id'])['wap'].apply(log_return)
    
    # Calculate realized volatility
    booklog_df = book_df[~book_df['log_return'].isnull()]
    realvol_df =  pd.DataFrame(booklog_df.groupby(['time_id'])['log_return'].agg(realized_volatility)).reset_index()
    realvol_df = realvol_df.rename(columns = {'log_return':'realized_vol'})
       
    realvol_df['stock_id'] = stock
    
    # Group by time ids
    temp = book_df.groupby(['time_id']).agg(
            # WAP
            wap_mean=pd.NamedAgg(column='wap', aggfunc="mean"),
            wap_median=pd.NamedAgg(column='wap', aggfunc="median"),
            wap_max=pd.NamedAgg(column='wap', aggfunc="max"),
            wap_min=pd.NamedAgg(column='wap', aggfunc="min"),
            wap2_mean=pd.NamedAgg(column='wap2', aggfunc="mean"),
            wap2_median=pd.NamedAgg(column='wap2', aggfunc="median"),
            wap2_max=pd.NamedAgg(column='wap2', aggfunc="max"),
            wap2_min=pd.NamedAgg(column='wap2', aggfunc="min"),
        
            # Spread
            spread_mean=pd.NamedAgg(column='spread', aggfunc="mean"),
            spread_median=pd.NamedAgg(column='spread', aggfunc="median"),
            spread_max=pd.NamedAgg(column='spread', aggfunc="max"),
            spread_min=pd.NamedAgg(column='spread', aggfunc="min"),
            spread2_mean=pd.NamedAgg(column='spread2', aggfunc="mean"),
            spread2_median=pd.NamedAgg(column='spread2', aggfunc="median"),
            spread2_max=pd.NamedAgg(column='spread2', aggfunc="max"),
            spread2_min=pd.NamedAgg(column='spread2', aggfunc="min"),
        
            # Bid-ask diff
            bid_ask_diff_mean=pd.NamedAgg(column='bid_ask_diff', aggfunc="mean"),
            bid_ask_diff_median=pd.NamedAgg(column='bid_ask_diff', aggfunc="median"),
            bid_ask_diff_max=pd.NamedAgg(column='bid_ask_diff', aggfunc="max"),
            bid_ask_diff_min=pd.NamedAgg(column='bid_ask_diff', aggfunc="min"),
            bid_ask_diff2_mean=pd.NamedAgg(column='bid_ask_diff2', aggfunc="mean"),
            bid_ask_diff2_median=pd.NamedAgg(column='bid_ask_diff2', aggfunc="median"),
            bid_ask_diff2_max=pd.NamedAgg(column='bid_ask_diff2', aggfunc="max"),
            bid_ask_diff2_min=pd.NamedAgg(column='bid_ask_diff2', aggfunc="min"),

            bid_ask_diff_wap_ratio_mean=pd.NamedAgg(column='bid_ask_diff_wap_ratio', aggfunc="mean"),
            bid_ask_diff_wap2_ratio_mean=pd.NamedAgg(column='bid_ask_diff_wap2_ratio', aggfunc="mean"),
            bid_ask_diff2_wap2_ratio_mean=pd.NamedAgg(column='bid_ask_diff2_wap2_ratio', aggfunc="mean"),
        
            ).reset_index(drop=False)
    
    
    book_feats_df = pd.merge(temp, realvol_df, on="time_id", how="left") 
    book_feats_df = book_feats_df.reset_index(drop=True)
    
    return book_feats_df

In [ ]:
stockLst = train_df['stock_id'].unique().tolist()

with concurrent.futures.ProcessPoolExecutor() as executor:
    results = list(tqdm(executor.map(order_book_features, stockLst), total=len(stockLst)))    

In [ ]:
opt_train_df = pd.concat(results)
opt_train_df.shape

# Recreate metrics from starter notebook

https://www.kaggle.com/jiashenliu/introduction-to-financial-concepts-and-data#Naive-prediction:-using-past-realized-volatility-as-target

In [ ]:
opt_train_df = pd.merge(opt_train_df, train_df, on=['stock_id','time_id'], how="left")

opt_train_df.head()

In [ ]:
from sklearn.metrics import r2_score
def rmspe(y_true, y_pred):
    return  (np.sqrt(np.mean(np.square((y_true - y_pred) / y_true))))
R2 = round(r2_score(y_true = opt_train_df['target'], y_pred = opt_train_df['realized_vol']),3)
RMSPE = round(rmspe(y_true = opt_train_df['target'], y_pred = opt_train_df['realized_vol']),3)
print(f'Performance of the naive prediction: R2 score: {R2}, RMSPE: {RMSPE}')

# Save file

In [ ]:
opt_train_df.to_csv("optiver_train1.csv", index=False)

In [ ]:
!ls -hlt /kaggle/working/